# Chunking e embedding

## Carregamento de bibliotecas

In [1]:
from  langchain_community.document_loaders import PyPDFLoader
import os
import warnings
from dotenv import load_dotenv
import httpx

warnings.filterwarnings("ignore")
http_client = httpx.Client(verify=False)

C:\Users\francisco.bneto\AppData\Local\Temp\ipykernel_42480\719751671.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from  langchain_community.document_loaders import PyPDFLoader
c:\Users\francisco.bneto\AppData\Local\miniconda3\envs\genai-py314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.chdir(r'c:\Users\francisco.bneto\Documents\gen-ai-formation')
print(os.getcwd())

c:\Users\francisco.bneto\Documents\gen-ai-formation


In [3]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
print("Chave carregada:", api_key[:10] + "...")

Chave carregada: sk-or-v1-c...


## Carregamento de documentos

In [4]:
documento = PyPDFLoader(
    "./documentos/GTB_gold_Nov23.pdf"
).load()

In [5]:
print(documento[0].page_content)

1 
Versão: novembro 2023 
2021 
 
 
 
Programa de Cartão da Edição Mastercard Gold 
 
Guia de Benefícios 
 
Informações importantes. Leia e guarde as informações. 
 
Este Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros 
e assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços 
estão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de 2023. 
Este Guia substitui qualquer guia ou comunicação de programa que você recebeu anteriormente. 
 
As informações contidas neste documento são apresentadas somente com propósito informativo. Não 
pretendem ser uma  descrição completa de todos os termos,  condições, limitações, exclusões ou outras 
disposições de qualquer programa ou benefícios de seguro fornecidos por, para, ou emitidos para a 
Mastercard. 
 
Nome do Representante: MASTERCARD DO BRASIL LTDA. CNPJ 01.248.201/0001-75. Nome da 
Seguradora: AIG Seguros Brasi

## Divisão do texto em chunkings

### RecursiveCharacterTextSplitter

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(documento)

In [7]:
len(chunks)

82

In [8]:
print(chunks[55].page_content)

Definições – Proteção de Preço 
 
“Compras Cobertas” significa itens comprados integralmente com o seu cartão Mastercard Gold™ e/ou 
adquiridos com pontos ganhos por meio de um Programa de Recompensas ligado ao seu cartão 
Mastercard Gold™. A lista dos bens elegíveis a Compras Cobertas encontra -se no item “Bens Elegíveis a 
este Seguro”. 
 
“Leilão (online ou ao vivo)” significa um local ou website onde itens são vendidos através de ofertas ou 
cotas; ou onde preços flutuam baseados no número de pessoas comprando, ou interessadas em comprar 
um produto. (Exemplos incluem, mas não estão limitados a Ebay, Ubid, Yahoo e leilões particulares 
ou públicos). 
 
“Anúncio Impresso” significa um anúncio exibido em jornais, revistas, impressos de lojas ou catálogo 
que declara o vendedor autorizado ou nome da loja, item (incluindo o  fornecedor e número  do 
modelo) e preço de venda. O anúncio deve ser publicado dentro de 30 dias a partir da data de compra


### CharacterTextSplitter

In [9]:
from langchain_text_splitters import CharacterTextSplitter

token_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=1000,
    chunk_overlap=100
)

token_chunks = token_splitter.split_documents(documento)

In [10]:
len(token_chunks)

24

### Tokenizer Splitter via Huggingface

In [11]:
from transformers import AutoTokenizer, AutoModel

emb_tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-small')
emb_model = AutoModel.from_pretrained('intfloat/multilingual-e5-small')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7874.14it/s]


In [12]:
hf_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=emb_tokenizer,
    chunk_size=512,
    chunk_overlap=50
)

hf_chunks = hf_splitter.split_documents(documento)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (720 > 512). Running this sequence through the model will result in indexing errors


In [13]:
len(hf_chunks)

24

### Splitter semântico

In [14]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    http_client=http_client
)

In [15]:
semantic_openai_splitter = SemanticChunker(embedding_model)


In [16]:
semantic_openai_chunks = semantic_openai_splitter.split_documents(documento)

In [17]:
len(semantic_openai_chunks)

46

In [18]:
print(semantic_openai_chunks[12].page_content)

Por favor, consulte seu Bilhete de Seguro para confirmar os limites segurados. Para os limites 
indicados no Bilhete de Seguro em dólares americanos (USD) os pagamentos das indenizações serão 
feitos na moeda local. (*) uma taxa de conversão não é uma taxa de câmbio. NOTA: 
 
Você pode obter o seu Bilhete de Seguro visitando o portal Bilhete de Seguro da Mastercard no endereço  
www.aig.com/Mastercard/pt. Esse documento deverá ser obrigatoriamente apresentado no caso de eventual 
ocorrência/ sinistro. O bilhete de seguro é válido para Bens Elegíveis comprados com o Cartão Mastercard Gold durante 
o período de 12 meses (contados do início de vigência do bilhete emitido). Isenção de Responsabilidade: As informações contidas neste documento são apresentadas


In [21]:
from langchain_huggingface import HuggingFaceEmbeddings

hf_emb_model = HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-small')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6064.74it/s]


In [22]:
semantic_hf_splitter = SemanticChunker(hf_emb_model)

semantic_hf_chunks = semantic_hf_splitter.split_documents(documento)

In [23]:
len(semantic_hf_chunks)

46

In [25]:
print(semantic_hf_chunks[44].page_content)

24 
Versão: novembro 2023 
2021 
 
 
 
Ações Jurídicas: Nenhuma ação legal deverá ser submetida para ressarcimento na Apólice até 60  
(sessenta) dias após a Empresa de Seguros ter fornecido a prova de perda, por escrito. Nenhuma ação poderá ser enviada depois de 3 (três) anos da data em que a prova de perda, por escrito, 
deve ser fornecida. Conformidade com os estatutos locais:  Qualquer provisão da Apólice que, em sua data de vigência, 
estiver em conflito com os estatutos do país no qual a apólice foi entregue ou emitida fica, por meio deste 
documento, alterada para estar em conformidade com os requisitos mínimos de tais estatutos. Arbitragem: Qualquer disputa relativa aos termos de quaisquer Apólices Master de seguro, incluindo 
qualquer dúvida com relação à sua existência, validade ou rescisão será referida e resolvida  por 
arbitragem e de acordo com os regulamentos/normas de arbitragem do país no qual sua conta de 
cartão Mastercard foi emitida. Confidencialidade e Segurança: 